# 05. Fixed-Cutoff Feature Build (21 days) (leakage fix)

**Purpose:** rebuild the customer features so that each one uses only events that happened **before a cutoff**, placed 21 days after each subscription starts. The original ETL aggregated features over each subscription's whole window, which ends at cancellation for churned customers (see `02_Machine_Learning.ipynb`).

**What this notebook does**
1. Builds a **filtered copy** of the raw inputs in `data/cutoff21/raw/`. The original raw files are read-only and are not modified.
2. Runs the **unchanged ETL feature functions** from `01_ETL_Pipeline.ipynb` on the filtered copy. The ETL notebook is read but not edited.
3. Checks that running the same functions on the unfiltered raw data reproduces the canonical Parquet exactly. This shows the reuse is faithful.
4. Saves `data/processed/customer_churn_dataset_cutoff21.parquet`. The canonical Parquet is not changed.

**Cutoff rule:** `T = start_date + 21 days`, per customer. Only events with timestamp `< T` are kept.

| Source | Treatment at cutoff |
|---|---|
| Subscription | `end_date` set to `T`, so the ETL window is `[start, T]`. `auto_renew` is the original outcome and is not changed. |
| Sales | Kept if `order_date < T`. Deliveries after `T` are blanked, so they count as in flight at `T`. |
| Payments, sale-products | Kept only for kept sales. |
| CRM reports | Kept if `received < T`. |
| Server log (web sessions) | Kept if `datetime < T`. |
| Product reviews | **Emptied.** Reviews have no date, so they cannot be placed before `T`. |
| Survey | Kept unchanged. It has **no timestamp**, so its timing relative to `T` is unknown. Used only in the B configuration. |

**Features excluded from modelling:** `duration_days` (it is set to 14 for everyone and carries no information), review features (no dates), and `has_reviews`.

## 1. Setup and the unchanged ETL functions

In [1]:
import os, shutil, sqlite3
from pathlib import Path
import nbformat
import numpy as np
import pandas as pd

CUTOFF_DAYS = 21

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for cand in [start, *start.parents]:
        if (cand / "data/raw/PerfectPetClub.db").exists():
            return cand
        if (cand / "PerfectPetClub/data/raw/PerfectPetClub.db").exists():
            return cand / "PerfectPetClub"
    raise FileNotFoundError("PerfectPetClub raw data not found")

PROJECT = find_project_root()
RAW_DIR = PROJECT / "data/raw"
CUT_DIR = PROJECT / "data/cutoff21"
FILT_RAW = CUT_DIR / "raw"
PROCESSED = PROJECT / "data/processed"
CANONICAL_PARQUET = PROCESSED / "customer_churn_dataset.parquet"
CUTOFF_PARQUET = PROCESSED / "customer_churn_dataset_cutoff21.parquet"
ETL_NOTEBOOK = PROJECT / "notebooks/01_ETL_Pipeline.ipynb"
CUT_DIR.mkdir(parents=True, exist_ok=True)

# Load only the ETL's function-definition cells, unchanged:
# cell 2 = imports, 15 = extraction, 16 = cleaning, 17 = features, 18 = integration.
etl_nb = nbformat.read(ETL_NOTEBOOK, as_version=4)
etl_ns = {}
for idx in [2, 15, 16, 17, 18]:
    exec(compile("".join(etl_nb.cells[idx].source), f"01_ETL cell {idx}", "exec"), etl_ns)
build_customer_dataset = etl_ns["build_customer_dataset"]
print("Loaded from 01_ETL_Pipeline.ipynb (read only): build_customer_dataset")
print("Project:", PROJECT)

Imports completed successfully:
  - pathlib (Path)
  - sqlite3
  - pandas (v2.3.3)
  - numpy (v2.5.4)
Extraction functions defined successfully:
  • extract_database_tables(db_path)
  • extract_csv_sources(raw_dir)
Cleaning functions defined successfully:
  • clean_crm_data(df_crm)
  • clean_server_logs(filepath, email_to_id_map)
  • clean_transaction_data(df_sales, df_payments)
  • clean_survey_data(df_survey, email_to_id_map)
Feature engineering functions defined successfully:
  • create_purchase_features
  • create_delivery_features
  • create_crm_features
  • create_review_features
  • create_engagement_features
  • create_survey_features
Integration function 'build_customer_dataset(raw_dir)' defined successfully.
Loaded from 01_ETL_Pipeline.ipynb (read only): build_customer_dataset
Project: /Users/atchyuthkarri/7COM2005-Data-Engineering/PerfectPetClub


/Users/atchyuthkarri/venvs/ml312/lib/python3.12/site-packages/nbformat/validator.py:434: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  _validate(nbdict, ref, version, version_minor, relax_add_props)


## 2. Build the filtered raw copy (cutoff = start + 21 days)

In [2]:
con = sqlite3.connect(f"file:{RAW_DIR / 'PerfectPetClub.db'}?mode=ro", uri=True)
sub = pd.read_sql_query("SELECT * FROM subscription", con)
cust = pd.read_sql_query("SELECT id, email, subscription_id FROM customer", con)
sale = pd.read_sql_query("SELECT * FROM sale", con)
sale_product = pd.read_sql_query("SELECT * FROM sale_product", con)
payment = pd.read_sql_query("SELECT * FROM payment", con)
sub_type = pd.read_sql_query("SELECT * FROM subscription_type", con)
product_review = pd.read_sql_query("SELECT * FROM product_review", con)
con.close()

# Per-subscription cutoff, then per-customer cutoff.
sub["cutoff_dt"] = pd.to_datetime(sub["start_date"]) + pd.Timedelta(days=CUTOFF_DAYS)
cust = cust.merge(sub[["id", "cutoff_dt"]].rename(columns={"id": "subscription_id"}), on="subscription_id", how="left")
cust_cutoff = dict(zip(cust["id"], cust["cutoff_dt"]))
email_cutoff = dict(zip(cust["email"], cust["cutoff_dt"]))
assert cust["cutoff_dt"].notna().all()

# Subscriptions: the ETL window ends at the cutoff. auto_renew (the outcome) is kept.
sub_f = sub.copy()
# Keep millisecond precision: starts carry ms, and dropping ms would shorten duration_days by a fraction.
sub_f["end_date"] = sub_f["cutoff_dt"].dt.strftime("%Y-%m-%d %H:%M:%S.%f").str[:-3]
sub_f = sub_f.drop(columns=["cutoff_dt"])

# Sales: keep only orders placed before the cutoff.
sale["order_dt"] = pd.to_datetime(sale["order_date"])
sale["cutoff_dt"] = sale["customer_id"].map(cust_cutoff)
sale_f = sale[sale["order_dt"] < sale["cutoff_dt"]].copy()
delivered_dt = pd.to_datetime(sale_f["delivered_date"])
late = delivered_dt > sale_f["cutoff_dt"]
sale_f.loc[late, "delivered_date"] = None   # not yet delivered at the cutoff
print(f"Sales kept: {len(sale_f)} of {len(sale)} | deliveries after cutoff blanked: {int(late.sum())}")
sale_f = sale_f[["id", "order_date", "expected_date", "delivered_date", "customer_id", "payment_id"]]

payment_f = payment[payment["id"].isin(sale_f["payment_id"])]
sale_product_f = sale_product[sale_product["sale_id"].isin(sale_f["id"])]

# Reviews have no date, so none can be placed before the cutoff: write an empty table.
product_review_f = product_review.iloc[0:0]

# CRM: keep reports received before the cutoff.
crm = pd.read_csv(RAW_DIR / "PerfectPetClub_CRM_report.csv")
crm_ts = pd.to_datetime(crm["recieved_datatime"])
crm_f = crm[crm_ts < crm["customer_id"].map(cust_cutoff)]
print(f"CRM rows kept: {len(crm_f)} of {len(crm)}")

# Server log: keep sessions before the cutoff, mapping login email to the customer.
log_path = RAW_DIR / "PerfectPetClub_server_log.csv"
out_log = CUT_DIR / "PerfectPetClub_server_log.csv"
kept_log, total_log, first = 0, 0, True
for chunk in pd.read_csv(log_path, chunksize=20000):
    total_log += len(chunk)
    ts = pd.to_datetime(chunk["datetime"])
    keep = ts < chunk["login"].map(email_cutoff)
    part = chunk[keep]
    kept_log += len(part)
    part.to_csv(out_log, mode="w" if first else "a", header=first, index=False)
    first = False
print(f"Server log rows kept: {kept_log} of {total_log}")

# Build the filtered inputs directory.
if FILT_RAW.exists():
    shutil.rmtree(FILT_RAW)
FILT_RAW.mkdir(parents=True)
db_out = FILT_RAW / "PerfectPetClub.db"
with sqlite3.connect(db_out) as oc:
    cust[["id", "email", "subscription_id"]].to_sql("customer", oc, index=False)
    sub_f.to_sql("subscription", oc, index=False)
    sub_type.to_sql("subscription_type", oc, index=False)
    sale_f.to_sql("sale", oc, index=False)
    sale_product_f.to_sql("sale_product", oc, index=False)
    payment_f.to_sql("payment", oc, index=False)
    product_review_f.to_sql("product_review", oc, index=False)
crm_f.to_csv(FILT_RAW / "PerfectPetClub_CRM_report.csv", index=False)
shutil.move(str(out_log), str(FILT_RAW / "PerfectPetClub_server_log.csv"))
shutil.copy2(RAW_DIR / "PerfectPetClub_survey.csv", FILT_RAW / "PerfectPetClub_survey.csv")
print("Filtered raw inputs written to:", FILT_RAW)

/var/folders/fd/85t_hlls27g226nv1ysgdk140000gn/T/ipykernel_5658/2418512561.py:12: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  sub["cutoff_dt"] = pd.to_datetime(sub["start_date"]) + pd.Timedelta(days=CUTOFF_DAYS)


Sales kept: 4191 of 23188 | deliveries after cutoff blanked: 351
CRM rows kept: 1101 of 7585


Server log rows kept: 12005 of 67249
Filtered raw inputs written to: /Users/atchyuthkarri/7COM2005-Data-Engineering/PerfectPetClub/data/cutoff21/raw


## 3. Reuse check: the unchanged ETL on the unfiltered data must reproduce the canonical Parquet

If this check fails, the reuse is not faithful and the cutoff features cannot be trusted.

In [3]:
full = build_customer_dataset(RAW_DIR)
canon = pd.read_parquet(CANONICAL_PARQUET)
shared = [c for c in canon.columns if c in full.columns]
full = full.set_index("customer_id")
canon_i = canon.set_index("customer_id")
mismatch = {}
for c in shared:
    if c == "customer_id":
        continue
    a, b = full.loc[canon_i.index, c], canon_i[c]
    if pd.api.types.is_numeric_dtype(a):
        ok = np.isclose(a.astype(float), b.astype(float), equal_nan=True, rtol=1e-9, atol=1e-9).all()
    else:
        ok = (a.astype(str).values == b.astype(str).values).all()
    if not ok:
        mismatch[c] = int((~np.isclose(a.astype(float), b.astype(float), equal_nan=True)).sum()) if pd.api.types.is_numeric_dtype(a) else "string"
print("Columns compared:", len(shared) - 1)
print("Mismatched columns:", mismatch if mismatch else "none")
assert not mismatch, "ETL reuse does not reproduce the canonical dataset"
print("OK: unchanged ETL reproduces the canonical Parquet on the unfiltered raw data.")

Columns compared: 35
Mismatched columns: none
OK: unchanged ETL reproduces the canonical Parquet on the unfiltered raw data.


## 4. Build the cutoff dataset

In [4]:
cut = build_customer_dataset(FILT_RAW).copy()
assert len(cut) == 221, len(cut)
assert (cut["duration_days"] == CUTOFF_DAYS).all()

# Counts with no events before the cutoff are true zeros, not missing.
COUNT_COLS = ["total_orders", "total_spend_pence", "total_items_purchased", "unique_products_purchased",
              "completed_deliveries", "unfulfilled_orders", "delayed_deliveries", "total_web_sessions"]
no_orders = int((cut["total_orders"].fillna(0) == 0).sum())
no_sessions = int(cut["total_web_sessions"].isna().sum())
no_crm = int((cut["total_crm_interactions"].fillna(0) == 0).sum())
for c in COUNT_COLS:
    cut[c] = cut[c].fillna(0)
cut["order_frequency_per_month"] = cut["total_orders"] / (CUTOFF_DAYS / 30.0)

# Rates that are undefined without events stay missing (NaN); the model imputes them from training data.
print(f"Customers with no orders before cutoff: {no_orders}")
print(f"Customers with no web sessions before cutoff (days_since_last_session NaN): {no_sessions}")
print(f"Customers with no CRM interactions before cutoff: {no_crm}")

# Drop features that are not cutoff-safe or are constant.
DROP = ["duration_days", "total_reviews", "avg_star_rating", "low_star_reviews_count",
        "high_star_reviews_count", "has_reviews"]
cut = cut.drop(columns=DROP)
assert "email" not in cut.columns
print("Final shape:", cut.shape, "| customer_id unique:", cut["customer_id"].is_unique)
print("Target distribution (auto_renew):", cut["auto_renew"].value_counts().sort_index().to_dict())
cut.to_parquet(CUTOFF_PARQUET, index=False)
print("Saved:", CUTOFF_PARQUET)

Customers with no orders before cutoff: 0
Customers with no web sessions before cutoff (days_since_last_session NaN): 0
Customers with no CRM interactions before cutoff: 61
Final shape: (221, 30) | customer_id unique: True
Target distribution (auto_renew): {0: 86, 1: 135}
Saved: /Users/atchyuthkarri/7COM2005-Data-Engineering/PerfectPetClub/data/processed/customer_churn_dataset_cutoff21.parquet


01_ETL cell 18:57: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
01_ETL cell 18:58: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


## 5. Compare with the canonical features

Shows how far the cutoff changes each feature. Counts and totals should be clearly lower, since they now cover 21 days rather than the whole subscription.

In [5]:
canon_cols = ["total_orders", "total_spend_pence", "total_crm_interactions", "total_web_sessions",
              "unfulfilled_orders", "delayed_deliveries", "days_since_last_session", "session_frequency_per_month"]
c2 = cut.set_index("customer_id")
comp = pd.DataFrame({
    "canonical_mean": canon_i[canon_cols].mean(),
    "cutoff21_mean": c2[canon_cols].mean(),
})
comp["ratio"] = (comp["cutoff21_mean"] / comp["canonical_mean"]).round(3)
display(comp.round(2))
print("Churn vs renewal mean total_orders at cutoff:")
print(cut.groupby("auto_renew")["total_orders"].mean().round(2).to_string())

,canonical_mean,cutoff21_mean,ratio
total_orders,104.92,18.96,0.18
total_spend_pence,479023.86,85177.81,0.18
total_crm_interactions,34.32,4.98,0.14
total_web_sessions,304.29,54.32,0.18
unfulfilled_orders,4.93,2.59,0.52
delayed_deliveries,3.87,0.51,0.13
days_since_last_session,15.58,1.69,0.11
session_frequency_per_month,56.03,77.60,1.38


Churn vs renewal mean total_orders at cutoff:
auto_renew
0    20.17
1    18.19


## 6. Outcome-proximity check

The original leakage was that churned customers' features covered activity near cancellation. This check confirms the cutoff dataset contains no such activity: each customer's most recent order or session is before `T`.

In [6]:
chk = sale_f.merge(cust[["id", "cutoff_dt"]], left_on="customer_id", right_on="id")
chk["order_dt"] = pd.to_datetime(chk["order_date"])
print("Orders at or after cutoff in filtered sales:", int((chk["order_dt"] >= chk["cutoff_dt"]).sum()))
print("Filtered server log: latest session before cutoff for each customer, max days-before-cutoff:")
logs = pd.read_csv(FILT_RAW / "PerfectPetClub_server_log.csv", usecols=["datetime", "login"])
logs["ts"] = pd.to_datetime(logs["datetime"])
logs["cut"] = logs["login"].map(email_cutoff)
print("Sessions at or after cutoff:", int((logs["ts"] >= logs["cut"]).sum()))

Orders at or after cutoff in filtered sales: 0
Filtered server log: latest session before cutoff for each customer, max days-before-cutoff:
Sessions at or after cutoff: 0
